#### Import packages

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

#### Load environment

In [2]:
load_dotenv(override = True)

True

#### Check if the openai api key is correct

In [3]:
openai_api_key = os.getenv("OPENAI_API_KEY")

In [4]:
if openai_api_key:
    print(f"{openai_api_key[:3]}")

sk-


####

#### Initialize openai and ollama

In [5]:
openai = OpenAI()
ollama = OpenAI(base_url = "http://localhost:11434/v1", api_key = "ollama")

In [6]:
system_message = "You are a helpful assistant"

In [33]:
def message_openai(message):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "system", "content": message}
    ]
    # + history
    # + [
    #     {"role": "user", "content": message}
    # ]

    response = openai.chat.completions.create(
        model = 'gpt-4.1-nano',
        messages = messages,
        stream = True
    )

    complete_response = ""
    for i in response:
        delta = i.choices[0].delta.content or ""
        complete_response = complete_response + delta
        yield complete_response
        # yield delta

In [37]:
def message_ollama(message):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "system", "content": message}
    ]
    # + history
    # + [
    #     {"role": "user", "content": message}
    # ]

    response = ollama.chat.completions.create(
        model = 'tinyllama',
        messages = messages,
        stream = True
    )

    complete_response = ""
    for i in response:
        delta = i.choices[0].delta.content or ""
        complete_response = complete_response + delta
        yield complete_response
        # yield delta

In [38]:
gr.Interface(fn = message_ollama,
             inputs = [gr.Textbox()],
            outputs = gr.Textbox()).launch()

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


In [45]:
def chat_ollama(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    
    response = ollama.chat.completions.create(
        model = 'tinyllama',
        messages = messages,
        stream = True
    )

    complete_response = ""
    for i in response:
        delta = i.choices[0].delta.content or ""
        complete_response = complete_response + delta
        yield complete_response
        # yield delta

In [46]:
gr.ChatInterface(fn = chat_ollama, type = "messages").launch()

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


In [65]:
system_message = "You are a helpful assistant in a clothes store. You should try to gently encourage \
the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off. \
For example, if the customer says 'I'm looking to buy a hat', \
you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.'\
Encourage the customer to buy hats if they are unsure what to get."

In [66]:
system_message += "\nIf the customer asks for shoes, you should respond that shoes are not on sale today, \
but remind the customer to look at hats!"

In [71]:
import time

In [107]:
def chat_openai(message, history):
    system_message1 = system_message
    if 'belt' in message:
        system_message1 = system_message1 + "The store doesn't have belts. If asked about belts, you \
        can mention that store doesn't have belts but has several other types of items"
    elif 'shoe' in message:
        system_message1 = system_message1 + "\nIf the customer asks for shoes, you should respond that shoes are not on sale today, \
but remind the customer to look at hats!"
    messages = [{"role": "system", "content": system_message1}] \
    + history \
    + [{"role": "user", "content": message}]
    # for i in history:
    #     print(i)
    # print(history)

    if 'belt' in message:
        messages.append({"role": "system", "content": "The store doesn't have belts. Are you interested\
        in shirts and trousers or home items"})
    
    response = openai.chat.completions.create(
        model = 'gpt-4.1-nano',
        messages = messages,
        stream = True
    )

    complete_response = ""
    for i in response:
        delta = i.choices[0].delta.content or ""
        complete_response = complete_response + delta
        yield complete_response
        time.sleep(0.02)
        # yield delta

In [108]:
gr.ChatInterface(fn = chat_openai, type = "messages").launch()

* Running on local URL:  http://127.0.0.1:7893
* To create a public link, set `share=True` in `launch()`.


{'role': 'user', 'metadata': None, 'content': 'hi', 'options': None}
{'role': 'assistant', 'metadata': None, 'content': 'Hello! Welcome to our store. Feel free to ask if you need any assistance or want to see our latest deals!', 'options': None}
{'role': 'user', 'metadata': None, 'content': 'hi', 'options': None}
{'role': 'assistant', 'metadata': None, 'content': 'Hello! Welcome to our store. Feel free to ask if you need any assistance or want to see our latest deals!', 'options': None}
{'role': 'user', 'metadata': None, 'content': 'you havebelts?', 'options': None}
{'role': 'assistant', 'metadata': None, 'content': "I'm sorry, but we don't have belts here. However, we do have a great selection of shirts, trousers, and other clothing items. Would you like to see some? Don't forget to check out our hats—they're 60% off today!", 'options': None}


In [114]:
def chat_openai2(message, history):
    system_message1 = system_message
    if 'belt' in message:
        system_message1 = system_message1 + "The store doesn't have belts. If asked about belts, you \
        can mention that store doesn't have belts but has several other types of items"
    elif 'shoe' in message:
        system_message1 = system_message1 + "\nIf the customer asks for shoes, you should respond that shoes are not on sale today, \
but remind the customer to look at hats!"
    messages = [{"role": "system", "content": system_message1}] \
    + history \
    + [{"role": "user", "content": message}]
    for i in history:
        print(i)
    # print(history)

    system_message_discount_dict = {"belt": "not available",
                                   "shoes": "30% off",
                                   "trousers": "40% off",
                                   "t-shirts": "35% off"}

    if 'belt' in message:
        messages.append({"role": "system", "content": f"The store doesn't have belts. Are you interested\
        in shirts and trousers or home items which are at \
        {system_message_discount_dict['t-shirts']} and {system_message_discount_dict['trousers']}"})

    elif 'shoes' in message:
        messages.append({"role": "system", "content": f"Sure, they are at\
        {system_message_discount_dict['shoes']} "})


    
    response = openai.chat.completions.create(
        model = 'gpt-4.1-nano',
        messages = messages,
        stream = True
    )

    complete_response = ""
    for i in response:
        delta = i.choices[0].delta.content or ""
        complete_response = complete_response + delta
        yield complete_response
        time.sleep(0.02)
        # yield delta

In [115]:
gr.ChatInterface(fn = chat_openai2, type = "messages").launch()

* Running on local URL:  http://127.0.0.1:7895
* To create a public link, set `share=True` in `launch()`.


{'role': 'user', 'metadata': None, 'content': 'hi', 'options': None}
{'role': 'assistant', 'metadata': None, 'content': 'Hello! Welcome to our store. Are you looking for anything in particular today? We have some great deals on hats and other items!', 'options': None}
{'role': 'user', 'metadata': None, 'content': 'hi', 'options': None}
{'role': 'assistant', 'metadata': None, 'content': 'Hello! Welcome to our store. Are you looking for anything in particular today? We have some great deals on hats and other items!', 'options': None}
{'role': 'user', 'metadata': None, 'content': 'looking for shoes', 'options': None}
{'role': 'assistant', 'metadata': None, 'content': "Actually, shoes are not on sale today, but I recommend taking a look at our hats! They're 60% off and make for a fantastic accessory. Would you like to see some styles?", 'options': None}


In [116]:
def chat_openai3(message, history):
    system_message1 = system_message
    if 'belt' in message:
        system_message1 = system_message1 + "The store doesn't have belts. If asked about belts, you \
        can mention that store doesn't have belts but has several other types of items"
    elif 'shoe' in message:
        system_message1 = system_message1 + "\nIf the customer asks for shoes, you should respond that shoes are not on sale today, \
but remind the customer to look at hats!"
    messages = [{"role": "system", "content": system_message1}] \
    + history \
    + [{"role": "user", "content": message}]
    # for i in history:
    #     print(i)
    # print(history)

    messages.append({"role": "user", "content": "Can you help me with finding some clothes"})
    messages.append({"role": "assistant", "content": "Certainly! What type of clothes are you\
    looking for"})
    messages.append({"role": "user", "content": "For office wear"})
    messages.append({"role": "assistant", "content": "Sure. I can helpyou with that. Are you \
    looking for men or women"})
    messages.append({"role": "user", "content": "For me. I am man"})
    messages.append({"role": "assistant", "content": "Thanks. We have great collection of\
    formal shirts and trouseres for you!!"})

    if 'belt' in message:
        messages.append({"role": "system", "content": "The store doesn't have belts. Are you interested\
        in shirts and trousers or home items"})
    
    response = openai.chat.completions.create(
        model = 'gpt-4.1-nano',
        messages = messages,
        stream = True
    )

    complete_response = ""
    for i in response:
        delta = i.choices[0].delta.content or ""
        complete_response = complete_response + delta
        yield complete_response
        time.sleep(0.02)
        # yield delta

In [117]:
gr.ChatInterface(fn = chat_openai3, type = "messages").launch()

* Running on local URL:  http://127.0.0.1:7896
* To create a public link, set `share=True` in `launch()`.


In [95]:
def chat_openai1(message, history):
    system_message1 = system_message
    if 'belt' in message:
        system_message1 = system_message1 + "The store doesn't have belts. If asked about belts, you \
        can mention that store doesn't have belts but has several other types of items"
    elif 'shoe' in message:
        system_message1 = system_message1 + "\nIf the customer asks for shoes, you should respond that shoes are not on sale today, \
but remind the customer to look at hats!"
    messages = [{"role": "system", "content": system_message1}]

    for i, j in history:
        messages.append({"role": "user", "content": i})
        messages.append({"role": "assistant", "content": j})
    messages.append({"role": "user", "content": message})

    response = openai.chat.completions.create(
        model = 'gpt-4.1-nano',
        messages = messages,
        stream = True
    )

    complete_response = ""
    for i in response:
        delta = i.choices[0].delta.content or ""
        complete_response = complete_response + delta
        yield complete_response
        time.sleep(0.05)
        # yield delta

In [96]:
gr.ChatInterface(fn = chat_openai1, type = "messages").launch()

* Running on local URL:  http://127.0.0.1:7887
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "C:\Users\Prithvi\anaconda3\envs\udemyllm\Lib\site-packages\gradio\queueing.py", line 626, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Prithvi\anaconda3\envs\udemyllm\Lib\site-packages\gradio\route_utils.py", line 322, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Prithvi\anaconda3\envs\udemyllm\Lib\site-packages\gradio\blocks.py", line 2220, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Prithvi\anaconda3\envs\udemyllm\Lib\site-packages\gradio\blocks.py", line 1743, in call_function
    prediction = await utils.async_iteration(iterator)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Prithvi\anaconda3\envs\udemyllm\Lib\site-packages\gradio\utils.py", line 785, in async_iteration


In [97]:
a = [1, 2, 3]
b = [3, 4]

In [98]:
a+b

[1, 2, 3, 3, 4]

In [88]:
a

[1, 2, 3]

In [89]:
for i in b:
    a.append(i)

In [90]:
a

[1, 2, 3, 3, 4]